## Creating a small ```DataFrame```

In [4]:
import pandas as pd

df = pd.DataFrame({
    "run_id": ["run_001", "run_002", "run_003"],
    "temperature_K": [300.1, 301.0, 299.7],
    "field_T": [0.15, 0.20, 0.10],
    "signal": [12.4, 13.1, 11.8],
})

print(df)

    run_id  temperature_K  field_T  signal
0  run_001          300.1     0.15    12.4
1  run_002          301.0     0.20    13.1
2  run_003          299.7     0.10    11.8


In [7]:
signal = df["signal"]

print(signal)
print(signal.index)
print(signal.values)

0    12.4
1    13.1
2    11.8
Name: signal, dtype: float64
RangeIndex(start=0, stop=3, step=1)
[12.4 13.1 11.8]


## Lazy queries are analysis plans

In [3]:
import os
import polars as pl

os.makedirs("data/processed", exist_ok=True)

df = pl.DataFrame(
    {
        "run_id": [1, 1, 1, 2, 2, 3],
        "chi2": [1.2, 2.1, 3.0, 0.8, 2.7, 1.5],
        "signal": [10.0, 12.0, 20.0, 5.0, 9.0, 30.0],
        "exposure_s": [2.0, 3.0, 4.0, 1.0, 3.0, 10.0],
    }
)

df.write_parquet("data/processed/events.parquet")

q = (
    pl.scan_parquet("data/processed/events.parquet")
      .filter(pl.col("chi2") < 2.5)
      .with_columns(
          signal_rate=pl.col("signal") / pl.col("exposure_s")
      )
      .group_by("run_id")
      .agg(
          n_events=pl.len(),
          mean_rate=pl.col("signal_rate").mean(),
      )
)

print(q.explain())

print(q.collect())

AGGREGATE[maintain_order: false]
  [len().alias("n_events"), col("signal_rate").mean().alias("mean_rate")] BY [col("run_id")]
  FROM
  simple π 2/2 ["run_id", "signal_rate"]
     WITH_COLUMNS:
     [[(col("signal")) / (col("exposure_s"))].alias("signal_rate")] 
      simple π 3/3 ["run_id", "signal", ... 1 other column]
        Parquet SCAN [data/processed/events.parquet]
        PROJECT 4/4 COLUMNS
        SELECTION: [(col("chi2")) < (2.5)]
        ESTIMATED ROWS: 6
shape: (3, 3)
┌────────┬──────────┬───────────┐
│ run_id ┆ n_events ┆ mean_rate │
│ ---    ┆ ---      ┆ ---       │
│ i64    ┆ u32      ┆ f64       │
╞════════╪══════════╪═══════════╡
│ 3      ┆ 1        ┆ 3.0       │
│ 1      ┆ 2        ┆ 4.5       │
│ 2      ┆ 1        ┆ 5.0       │
└────────┴──────────┴───────────┘
